# Multi-Hop Architecture Example

In [0]:
%pip install Faker
%pip install --upgrade typing-extensions


In [0]:
dbutils.library.restartPython()

In [0]:
from pyspark.sql import SparkSession
import datetime
import re
import os
import time
import pyspark
from pyspark.sql.types import StructType, StructField, BooleanType, DoubleType, LongType
from pyspark.sql.types import StringType
from pyspark.sql.types import DateType
from pyspark.sql.types import IntegerType
from pyspark.sql import Row
from datetime import date
from faker import Faker
import random
import numpy as np
from datetime import timedelta



In [0]:
def generate_dummy_customers(n=1000):
    fake = Faker()
    customer_list = []
    for i in range(n):
        customer_list.append(
            {
                'customer_id': fake.uuid4(),
                'customer_name': fake.name(),
                'customer_email': fake.email()
            }
        )
    

    return customer_list


In [0]:
def generate_dummy_books(n=1000):
    fake = Faker()
    book_list = []
    for i in range(n):
        book_list.append(
            {
                'book_id': fake.uuid4(),
                'book_title': fake.name(),
                'book_author': fake.name(),
                'book_category': fake.job(),
                'book_price': random.randint(10, 100)
            }
        )
    

    return book_list

In [0]:
def pull_random_customer(customer_data):
    item = random.choice(customer_data)

    return item['customer_id']


def pull_random_book(book_data):
    item = random.choice(book_data)

    return item['book_id']


def generate_dummy_order(customer_data, book_data, n=100):
    order_list = []
    fake = Faker()
    for i in range(n):
        customer_id = pull_random_customer(customer_data)
        book_id = pull_random_book(book_data)


        order_list.append({
            'order_id': fake.uuid4(),
            'customer_id': customer_id,
            'book_id': book_id,
            'quantity': random.randint(1, 10),
            'order_date': fake.date_this_year().strftime("%Y-%m-%d")
          #  'order_date': datetime.datetime.now().strftime('%Y-%m-%d')
        })
    
    return order_list

In [0]:
customers_data = generate_dummy_customers()
books_data = generate_dummy_books()
orders_data = generate_dummy_order(customers_data, books_data)

print(orders_data[0])

In [0]:

spark = SparkSession.builder \
        .appName('Multi Hop Architecture') \
    .getOrCreate()






#spark.conf.set("spark.sql.legacy.parquet.nanosAsLong", "true")
spark.conf.set("spark.sql.session.timeZone", "UTC")
#spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "true")
#spark._jsc.hadoopConfiguration().set(f"fs.azure.account.key.{STORAGE_ACCOUNT}.dfs.core.windows.net",f'{STORAGE_ACCOUNT_KEY}')

In [0]:


def create_and_load_files(n_customers=1000, n_books=1000, n_orders=10000):
    customers_data = generate_dummy_customers(n_customers)
    books_data = generate_dummy_books(n_books)
    orders_data = generate_dummy_order(customers_data, books_data, n_orders)

    customers_data = spark.createDataFrame(customers_data)
    books_data = spark.createDataFrame(books_data)
    orders_data = spark.createDataFrame(orders_data)


    customers_data \
            .write \
            .format("parquet") \
            .option("header", "true")\
            .mode("append") \
            .save("/Volumes/mlb_demo/default/file_examples/library_example/customers")
    
    books_data \
            .write \
            .format("parquet") \
            .option("header", "true")\
            .mode("append") \
            .save("/Volumes/mlb_demo/default/file_examples/library_example/books")
    orders_data \
            .write \
            .format("parquet") \
            .option("header", "true")\
            .mode("append") \
            .save("/Volumes/mlb_demo/default/file_examples/library_example/orders_data")

In [0]:
create_and_load_files()

## Stream Raw Data to a tmp table

In [0]:
schema_name = 'gen_prod.analytics_mee_silver'

In [0]:
spark \
    .readStream \
    .format("cloudFiles") \
    .option("cloudFiles.format", "parquet") \
    .option("cloudFiles.schemaLocation", "/Volumes/mlb_demo/default/file_examples/library_example/schema") \
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")\
    .load("/Volumes/mlb_demo/default/file_examples/library_example/orders_data")\
    .createOrReplaceTempView("orders")


In [0]:
%sql

CREATE OR REPLACE TEMPORARY VIEW orders_tmp AS
SELECT *, current_timestamp() as arrival_time, input_file_name() as source_file
FROM orders;

In [0]:

display(spark.sql("SELECT * FROM orders_tmp"),checkpointLocation='/Volumes/mlb_demo/default/file_examples/library_example/checkpointD' )


## Write it back to a spark delta table Bronze

In [0]:
spark.table("orders_tmp") \
    .writeStream \
    .format("delta") \
    .option("checkpointLocation", "/Volumes/mlb_demo/default/file_examples/library_example/checkpointB") \
    .outputMode("append") \
    .table("orders_bronze") 

In [0]:
%sql
select *
from {schema_name}.dim_acct

In [0]:
%sql


SELECT COUNT(*) FROM orders_bronze

In [0]:
create_and_load_files()

## Silver Layer

In [0]:
spark \
    .read \
    .format('parquet') \
    .option("header", "true") \
    .load("/Volumes/mlb_demo/default/file_examples/library_example/customers")\
    .createOrReplaceTempView("customer_info")


In [0]:
%sql

CREATE OR REPLACE TEMPORARY VIEW CUSTOMER_ORDERS AS
SELECT bronze.order_id ,customer.customer_id, customer.customer_name, customer.customer_email, bronze.book_id, bronze.arrival_time, bronze.source_file, bronze.order_date, bronze.quantity
FROM orders_bronze bronze
left join customer_info customer on customer.customer_id = bronze.customer_id


In [0]:
%sql

SELECT * FROM CUSTOMER_ORDERS;